# LED score logcat analysis

Loads Android Studio Logcat text, extracts `LedScoresCsv` payload lines, builds a table, saves a cleaned CSV, and plots one interactive Plotly chart.


In [1]:
from pathlib import Path
import re

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

LOG_PATH = Path("led_logcat.txt")
if not LOG_PATH.exists():
    LOG_PATH = Path("analysis/data/led_logcat.txt")
if not LOG_PATH.exists():
    LOG_PATH = Path("../data/led_logcat.txt")
CLEAN_CSV_PATH = LOG_PATH.with_name("led_scores_clean.csv")

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 20)

## Parse logcat

Android Studio prefixes lines with wall-clock time, pid, tag, package, and level. The parser extracts both that wall-clock time and the CSV payload after log level `D`. Raw CSV copied from `adb logcat -v raw` is also accepted, but then relative time falls back to `timestampNs`.


In [2]:
payload_re = re.compile(
    r"^(?:(?P<logTime>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}\.\d{3}).*?)?"
    r"\bLedScoresCsv\b.*?\bD\s+(?P<payload>(?:\d{6,}.*)|(?:timestampNs,.*))$"
)
raw_data_re = re.compile(r"^\d{6,},[01],")
header = "timestampNs,detected,s0,s1,s2,s3,s4,event"

records = []
for line in LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines():
    line = line.strip()
    match = payload_re.search(line)
    if match:
        payload = match.group("payload").strip()
        log_time = match.group("logTime")
    elif line == header or raw_data_re.match(line):
        payload = line
        log_time = None
    else:
        continue
    if payload == header:
        continue
    records.append((log_time, payload))

if not records:
    raise ValueError(f"No LedScoresCsv payload rows found in {LOG_PATH.resolve()}")

rows = []
for log_time, payload in records:
    parts = payload.split(",")
    if len(parts) < 8:
        parts += [""] * (8 - len(parts))
    if len(parts) > 8:
        parts = parts[:7] + [",".join(parts[7:])]
    rows.append([log_time, *parts])

df = pd.DataFrame(
    rows,
    columns=[
        "logTime",
        "timestampNs",
        "detected",
        "s0",
        "s1",
        "s2",
        "s3",
        "s4",
        "event",
    ],
)
df["logTime"] = pd.to_datetime(df["logTime"], errors="coerce")
df["timestampNs"] = pd.to_numeric(df["timestampNs"], errors="coerce").astype("Int64")
df["detected"] = pd.to_numeric(df["detected"], errors="coerce").fillna(0).astype(int)
score_cols = ["s0", "s1", "s2", "s3", "s4"]
for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["event"] = df["event"].replace("", pd.NA)
df = df.dropna(subset=["timestampNs"]).reset_index(drop=True)

if df["logTime"].notna().any():
    first_time = df.loc[df["logTime"].notna(), "logTime"].iloc[0]
    df["t_sec"] = (df["logTime"] - first_time).dt.total_seconds()
    missing_time = df["t_sec"].isna()
    if missing_time.any():
        fallback = (
            df.loc[missing_time, "timestampNs"].astype("float64")
            - float(df["timestampNs"].iloc[0])
        ) / 1e9
        df.loc[missing_time, "t_sec"] = fallback
else:
    df["t_sec"] = (
        df["timestampNs"].astype("float64") - float(df["timestampNs"].iloc[0])
    ) / 1e9

df["frame_dt_ms"] = df["t_sec"].diff() * 1000.0

df.to_csv(CLEAN_CSV_PATH, index=False)
print(f"log path: {LOG_PATH.resolve()}")
print(f"parsed rows: {len(df)}")
print(f"detected rows: {int(df.detected.sum())}")
print(f"events: {df.event.notna().sum()}")
print(f"time range: {df.t_sec.min():.3f}s .. {df.t_sec.max():.3f}s")
print(f"saved: {CLEAN_CSV_PATH.resolve()}")
df.head(12)

log path: C:\Users\shich\Src\bottleneck_code_transmission\analysis\data\led_logcat.txt
parsed rows: 447
detected rows: 359
events: 96
time range: 0.000s .. 14.836s
saved: C:\Users\shich\Src\bottleneck_code_transmission\analysis\data\led_scores_clean.csv


,logTime,timestampNs,detected,s0,s1,s2,s3,s4,event,t_sec,frame_dt_ms
0,2026-06-12 12:14:52.261,111896829639957,0,NaN,NaN,NaN,NaN,NaN,NaN,0.000,NaN
1,2026-06-12 12:14:52.293,111896896290999,0,NaN,NaN,NaN,NaN,NaN,NaN,0.032,32.0
2,2026-06-12 12:14:52.317,111896929616624,0,NaN,NaN,NaN,NaN,NaN,NaN,0.056,24.0
3,2026-06-12 12:14:52.347,111896962942041,0,NaN,NaN,NaN,NaN,NaN,NaN,0.086,30.0
4,2026-06-12 12:14:52.378,111896996267457,0,NaN,NaN,NaN,NaN,NaN,NaN,0.117,31.0
5,2026-06-12 12:14:52.419,111897029593082,0,NaN,NaN,NaN,NaN,NaN,NaN,0.158,41.0
6,2026-06-12 12:14:52.448,111897062918499,0,NaN,NaN,NaN,NaN,NaN,NaN,0.187,29.0
7,2026-06-12 12:14:52.488,111897096244124,0,NaN,NaN,NaN,NaN,NaN,NaN,0.227,40.0
8,2026-06-12 12:14:52.518,111897129569541,0,NaN,NaN,NaN,NaN,NaN,NaN,0.257,30.0
9,2026-06-12 12:14:52.547,111897162894957,0,NaN,NaN,NaN,NaN,NaN,NaN,0.286,29.0


## Quick stats


In [3]:
display(df[["t_sec", "frame_dt_ms", "detected", *score_cols, "event"]].describe())
display(df[df["event"].notna()][["t_sec", "logTime", "timestampNs", "event"]].head(80))

,t_sec,frame_dt_ms,detected,s0,s1,s2,s3,s4
count,447.000000,446.000000,447.000000,359.000000,359.000000,359.000000,359.000000,359.000000
mean,7.423468,33.264574,0.803132,0.857516,0.768469,0.814455,0.786017,0.840073
std,4.304594,7.819132,0.398077,0.524870,0.589145,0.549129,0.571793,0.504448
min,0.000000,4.000000,0.000000,0.074200,-0.198100,-0.082700,-0.101000,0.133500
25%,3.710000,28.000000,1.000000,0.353500,0.122450,0.225550,0.159400,0.375600
50%,7.421000,33.000000,1.000000,1.319100,1.245300,1.238600,1.238700,1.318300
75%,11.137500,38.000000,1.000000,1.320950,1.319450,1.319100,1.319300,1.320100
max,14.836000,58.000000,1.000000,1.348500,1.347800,1.340200,1.344200,1.325900


,t_sec,logTime,timestampNs,event
88,2.947,2026-06-12 12:14:55.208,111899795609541,00001
92,3.062,2026-06-12 12:14:55.323,111899928911624,00011
96,3.190,2026-06-12 12:14:55.451,111900062213499,00111
99,3.304,2026-06-12 12:14:55.565,111900162189957,01111
103,3.435,2026-06-12 12:14:55.696,111900295492041,11111
107,3.567,2026-06-12 12:14:55.828,111900428794124,11110
111,3.689,2026-06-12 12:14:55.950,111900562095999,11100
115,3.821,2026-06-12 12:14:56.082,111900695398082,11000
118,3.932,2026-06-12 12:14:56.193,111900795374541,10000
123,4.096,2026-06-12 12:14:56.357,111900962002041,00010


## LED Scores

One compact interactive Plotly chart with only LED score curves. Use the legend to enable/disable individual LEDs.


In [4]:
ON_THRESHOLD = 0.72
OFF_THRESHOLD = 0.48
SCORE_Y_RANGE = [-0.10, 1.42]
LED_COLORS = ["#55A7FF", "#65D6A4", "#FFD166", "#FF7A90", "#C99CFF"]

plot_df = df.copy()
for col in score_cols:
    plot_df.loc[plot_df["detected"] == 0, col] = pd.NA

fig = go.Figure()
for col, color in zip(score_cols, LED_COLORS):
    fig.add_trace(
        go.Scatter(
            x=plot_df["t_sec"],
            y=plot_df[col],
            mode="lines",
            name=col,
            line=dict(width=1.8, color=color),
            connectgaps=False,
            hovertemplate=f"{col}: %{{y:.4f}}<br>t=%{{x:.3f}}s<extra></extra>",
        )
    )

fig.add_hline(y=ON_THRESHOLD, line_dash="dash", line_color="black", opacity=0.65, annotation_text="ON", annotation_position="top left")
fig.add_hline(y=OFF_THRESHOLD, line_dash="dash", line_color="gray", opacity=0.65, annotation_text="OFF", annotation_position="top left")

fig.update_layout(
    title="LED scores over time",
    xaxis_title="time from capture start, seconds",
    yaxis_title="score",
    width=780,
    height=400,
    yaxis=dict(range=SCORE_Y_RANGE),
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=55, r=20, t=70, b=55),
)
fig.show()
